# Clean — MDS Bone Marrow Cell Dataset

**Vấn đề từ EDA**: 54 lớp, mất cân bằng 3.958 lần, nhiều lớp <20 ảnh.

**Quyết định**: Gộp các lớp <20 ảnh thành nhóm `Rare_Other` — giữ thông tin thay vì xóa hẳn. Không copy lại 25.067 ảnh (tốn 500MB+ dung lượng trùng lặp) — thay vào đó tạo **file manifest** ánh xạ `image_path -> lớp gốc -> lớp sau gộp`, dùng file này để load ảnh theo lớp mới khi train.

**Output**: `Clean_Data/tabular/mds_bone_marrow_manifest.csv`

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path(r'D:\AI_08_V1\Data\images\mds_bone_marrow\extracted')
OUT = Path(r'D:\AI_08_V1\Clean_Data\tabular')
IMG_EXT = {'.jpg', '.jpeg', '.png'}

# Gom toàn bộ ảnh: main/ và add/ của cùng 1 loại tế bào coi là cùng 1 lớp gốc
records = []
for sub in sorted(ROOT.rglob('*')):
    if sub.is_dir():
        cell_type = sub.name  # tên loại tế bào (bỏ qua main/add ở path cha)
        for f in sub.iterdir():
            if f.is_file() and f.suffix.lower() in IMG_EXT:
                records.append({'image_path': str(f), 'source_folder': sub.parent.name, 'original_class': cell_type})

manifest = pd.DataFrame(records)
print('Tổng số ảnh:', len(manifest))

class_counts = manifest['original_class'].value_counts()
print('Số lớp gốc:', len(class_counts))
rare_classes = class_counts[class_counts < 20].index.tolist()
print(f'Số lớp <20 ảnh sẽ gộp vào Rare_Other: {len(rare_classes)}')
print('Danh sách:', rare_classes)

Tổng số ảnh: 25009
Số lớp gốc: 32
Số lớp <20 ảnh sẽ gộp vào Rare_Other: 5
Danh sách: ['Monoblast', 'Megakaryocyte', 'Megaloblastic late erythroblast', 'Histiocyte', 'Dysplastic megakaryocyte']


In [2]:
# Tạo cột clean_class: gộp main/add cùng loại tế bào (đã làm ở bước gom trên) + gộp lớp hiếm
manifest['clean_class'] = manifest['original_class'].where(~manifest['original_class'].isin(rare_classes), 'Rare_Other')

new_counts = manifest['clean_class'].value_counts()
print('Số lớp sau khi gộp (main+add theo loại tế bào, và gộp lớp hiếm):', len(new_counts))
print(f'Tỷ lệ mất cân bằng mới (max/min): {new_counts.max()/new_counts.min():.0f} lần (so với 3.958 lần ban đầu)')
print('\nPhân bố lớp sau cùng:')
print(new_counts)

manifest.to_csv(OUT / 'mds_bone_marrow_manifest.csv', index=False)
print(f'\nĐã lưu manifest: {OUT / "mds_bone_marrow_manifest.csv"} ({len(manifest)} dòng)')

Số lớp sau khi gộp (main+add theo loại tế bào, và gộp lớp hiếm): 28
Tỷ lệ mất cân bằng mới (max/min): 187 lần (so với 3.958 lần ban đầu)

Phân bố lớp sau cùng:
clean_class
Mature lymphocyte             4112
Late erythroblast             3767
Smudge cell                   3127
Intermediate erythroblast     2310
Band neutrophil               2243
Segmented neutrophil          1803
Neutrophilic metamyelocyte    1347
Myeloblast                    1192
Neutrophilic myelocyte         873
Monocyte                       787
Early erythroblast             786
Promyelocyte                   472
Dysplastic erythroblast        464
Small megakaryocyte            294
Plasma cell                    239
Micromegakaryocyte             210
Dysplastic granulocyte         200
Eosinophilic metamyelocyte     156
Mitosis                        130
Segmented eosinophil           100
Segmented basophil              90
Band eosinophil                 59
Unclassified cell               55
Proerythroblast        